# [Chapter 14 — Vector-Borne Diseases, §14.6] Seasonal Vector Dynamics: Forcing $\mathcal{R}_0(t)$ Across the Year

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 14 — Vector-Borne Diseases
**Considerations developed:** 2 (short-term simulation assumptions), 13 (equilibrium vs invasion conflation)
**Estimated runtime:** ≤ 45 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_14_vector_borne/02_seasonal_dynamics.ipynb)


## What this notebook does

Adds a sinusoidal seasonality to the vector birth-and-death rate, capturing the temperature-driven annual cycle that drives mosquito populations in temperate and subtropical regions. The vector compartment cycles annually because vectors live only weeks; the human compartment, by contrast, is dominated by susceptible-depletion from the first outbreak. The notebook quantifies the resulting $\mathcal{R}_0(t)$ envelope and shows why a single annual-mean $\mathcal{R}_0$ misrepresents both invasion potential (Consideration~13) and short-term outbreak risk during the high-transmission season (Consideration~2).

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_14
from shared.verification import assert_within_tolerance
set_seed_chapter_14()
book_style()


## Seasonal vector mortality

$$\mu_v(t) = \bar\mu_v \left[1 + A \cos\left(\frac{2\pi t}{365}\right)\right]$$
with amplitude $A=0.5$. Vector mortality peaks in the cold season.

In [ ]:
from shared.parameters import baseline_chapter_14
import math
p = baseline_chapter_14()
p['t_end'] = 365.0 * 4
p['n_points'] = 8001
AMPL = 0.5

def mu_v_t(t, p, A=AMPL):
    return p['mu_v'] * (1 + A * np.cos(2*np.pi*t/365))

def vh_seasonal(y, t, p):
    SH, IH, RH, SV, IV = y
    a, b, c = p['a'], p['b'], p['c']
    mu_v = mu_v_t(t, p)
    gamma_h = p['gamma_h']
    MN = p['M_over_N']
    Lam_h = a * b * MN * IV
    Lam_v = a * c * IH
    return [-Lam_h*SH, Lam_h*SH - gamma_h*IH, gamma_h*IH,
            mu_v*(1-SV) - Lam_v*SV, Lam_v*SV - mu_v*IV]

y0 = [p['SH0'], p['IH0'], p['RH0'], p['SV0'], p['IV0']]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(vh_seasonal, y0, t, args=(p,))
SH, IH, RH, SV, IV = sol.T


## Annual outbreak structure

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(t/365, IH, color=BOOK_COLORS['infectious'], lw=1.5)
axes[0].set_ylabel('I_H (human prevalence)')
axes[0].set_title('Seasonal vector dynamics: human outbreak + susceptible depletion')
axes[0].grid(True, alpha=0.3)
axes[1].plot(t/365, mu_v_t(t, p), color=BOOK_COLORS['secondary'], lw=1.2)
axes[1].set_xlabel('time (years)')
axes[1].set_ylabel('vector mortality $\\mu_v(t)$')
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print('Note: a single closed human cohort produces one large outbreak; in real populations,')
print('demographic turnover or waning immunity replenishes susceptibles, producing the')
print('observed annual or multi-year cycles seen for dengue in Asia and Latin America.')


## Time-varying $\mathcal{R}_0(t)$ and the threshold conflation

Compute the instantaneous Ross-Macdonald $\mathcal{R}_0(t)$ from the seasonal $\mu_v(t)$. The annual-mean $\bar{\mathcal{R}}_0$ may exceed 1 even when invasion fails because of the geometry of the seasonal forcing (Floquet theory, beyond this notebook's scope).

In [ ]:
a, b, c, gamma_h, MN = p['a'], p['b'], p['c'], p['gamma_h'], p['M_over_N']
R0_t = np.sqrt(a**2 * b * c * MN / (mu_v_t(t, p) * gamma_h))
R0_mean = R0_t.mean()
R0_min = R0_t.min()
R0_max = R0_t.max()
print(f'R_0 range across the year: [{R0_min:.2f}, {R0_max:.2f}]')
print(f'Annual-mean R_0 (arithmetic): {R0_mean:.2f}')
print()
print('Note: a Floquet analysis would give the *true* invasion threshold,')
print('which is not the same as either the mean or any single-time R_0.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t/365, R0_t, color=BOOK_COLORS['primary'], lw=1.5)
ax.axhline(1, ls=':', color='gray')
ax.axhline(R0_mean, ls='--', color=BOOK_COLORS['highlight'], label='annual mean')
ax.set_xlabel('time (years)')
ax.set_ylabel('R_0(t)')
ax.set_title('Seasonal R_0(t) — single-number reports lose this signal (Consideration 13)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verification: confirm that R_0(t) genuinely cycles with annual period
amplitude = (R0_max - R0_min) / R0_mean
print(f'\nRelative amplitude of R_0(t): {amplitude:.2f}')
assert amplitude > 0.3, 'seasonal R_0 should vary substantially'
print('Verified: R_0(t) shows substantial seasonal variation that single-number reports erase.')


## What's next

- Chapter 18 (Dengue) uses real climate data instead of a sinusoid.
- Floquet stability analysis (book §11.6) gives the rigorous invasion threshold for periodic systems.
- The principle generalizes: any time-varying parameter (vaccination rollout, seasonal contact patterns) creates the same Consideration~13 issue with single-number $\mathcal{R}_0$ reports.